# 08 Bootstrap CIs and Report Tables

Create final metric tables and bootstrap confidence intervals from score files produced by earlier notebooks. Baseline CIs are generated from notebook 06 outputs; AE CIs are generated automatically if notebook 05 score files are available.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score


In [ ]:
REPO_ROOT = Path('..').resolve()
SCORE_DIR = REPO_ROOT / 'reports' / 'scores'
BASELINE_DIR = REPO_ROOT / 'reports' / 'baselines'
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'
FIGURE_DIR = REPO_ROOT / 'reports' / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

N_BOOTSTRAP = 1000
RANDOM_SEED = 42
AE_SCORE_COLUMNS = ['global_mse', 'global_mae', 'ver_max', 'ver_topk']


In [ ]:
def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, scores)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def bootstrap_intervals(y_true: np.ndarray, scores: np.ndarray, threshold: float, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_SEED) -> dict:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    rows = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        sampled_y = y_true[idx]
        sampled_scores = scores[idx]
        if len(np.unique(sampled_y)) < 2:
            continue
        rows.append(evaluate_at_threshold(sampled_y, sampled_scores, threshold))
    boot = pd.DataFrame(rows)
    intervals = {}
    for metric in ['pr_auc', 'f1', 'precision', 'recall']:
        intervals[f'{metric}_ci_low'] = float(boot[metric].quantile(0.025))
        intervals[f'{metric}_ci_high'] = float(boot[metric].quantile(0.975))
    intervals['bootstrap_n_effective'] = int(len(boot))
    return intervals


In [ ]:
baseline_scores_path = BASELINE_DIR / 'baseline_scores_turning.csv'
baseline_thresholds_path = THRESHOLD_DIR / 'baseline_thresholds_turning.json'
rows = []

if baseline_scores_path.exists() and baseline_thresholds_path.exists():
    baseline_scores = pd.read_csv(baseline_scores_path)
    with baseline_thresholds_path.open() as f:
        baseline_thresholds = json.load(f)
    for method, group in baseline_scores[baseline_scores['split'] == 'test'].groupby('method'):
        threshold = float(baseline_thresholds[method]['threshold'])
        y_true = group['target'].to_numpy()
        scores = group['score_value'].to_numpy()
        metrics = evaluate_at_threshold(y_true, scores, threshold)
        intervals = bootstrap_intervals(y_true, scores, threshold)
        rows.append({'dataset': 'turning', 'method': method, 'score': 'anomaly_score', 'threshold': threshold, **metrics, **intervals})

baseline_ci = pd.DataFrame(rows)
if not baseline_ci.empty:
    output_path = TABLE_DIR / 'metrics_turning_baselines_with_ci.csv'
    baseline_ci.to_csv(output_path, index=False)
    print(f'Wrote {output_path}')
baseline_ci


In [ ]:
ae_score_path = SCORE_DIR / 'ae_scores_turning.csv'
ae_threshold_path = THRESHOLD_DIR / 'ae_thresholds_turning.json'
ae_rows = []

if ae_score_path.exists() and ae_threshold_path.exists():
    scores = pd.read_csv(ae_score_path)
    with ae_threshold_path.open() as f:
        thresholds = json.load(f)
    test_scores = scores[scores['split'] == 'test'].copy()
    test_scores['target'] = (test_scores['label'] == 'chatter').astype(int)
    y_true = test_scores['target'].to_numpy()
    for score_name in AE_SCORE_COLUMNS:
        threshold = float(thresholds[score_name]['threshold'])
        score_values = test_scores[score_name].to_numpy()
        metrics = evaluate_at_threshold(y_true, score_values, threshold)
        intervals = bootstrap_intervals(y_true, score_values, threshold)
        ae_rows.append({'dataset': 'turning', 'method': 'cnn_ae', 'score': score_name, 'threshold': threshold, **metrics, **intervals})
else:
    print('AE score files not found; skipping AE confidence intervals.')

ae_ci = pd.DataFrame(ae_rows)
if not ae_ci.empty:
    output_path = TABLE_DIR / 'metrics_turning_ae_with_ci.csv'
    ae_ci.to_csv(output_path, index=False)
    print(f'Wrote {output_path}')
ae_ci


## Broaching Data

Add the broaching score file and threshold file here when the broaching dataset is available in the repository or mounted at a documented path.
